In [ ]:
!mkdir -p data
!cd data

!wget https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt
!wget https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt

!cd ..


In [1]:
import os
import regex as re
from time import time
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor

In [2]:
i2b_vocab: dict[int, bytes] = {x: bytes([x]) for x in range(256)}
# b2i_vocab: dict[bytes, int] = {bytes([x]): x for x in range(256)}

# print(f"{i2b_vocab}\n\n{b2i_vocab}")

In [3]:
train_data = "/kaggle/working/TinyStoriesV2-GPT4-train.txt"
with open(train_data, "r", encoding="utf-8") as f:
    text = f.read().split("<|endoftext|>")

In [4]:

class RegexTokenizer():
    
    def encode(self, text: str, merge_order: list[tuple[tuple[int, int], int]]) -> list[int]:
        text_ids = list(map(int, text.encode('utf-8')))
        for pair, target_idx in merge_order:
            text_ids_copy = text_ids.copy()
            for n in range(len(text_ids_copy)-1, 0, -1):
                if (text_ids_copy[n-1], text_ids_copy[n]) == pair:
                    text_ids_copy[n] = target_idx
                    del text_ids_copy[n-1]
            text_ids = text_ids_copy
            del text_ids_copy
        return text_ids

    def nested_decode(self, token_seq: list[list[int]], i2b_vocab: dict[int, bytes]) -> str:
        assert isinstance(token_seq, list) and all(isinstance(x, list) for x in token_seq)
        byte_string = [i2b_vocab[i] for seq in token_seq for i in seq]
        return b"".join(byte_string).decode('utf-8', errors='replace')

    def flattened_decode(self, token_seq: list[int], i2b_vocab: dict[int, bytes]) -> str:
        assert isinstance(token_seq, list) and all(type(x) is int for x in token_seq)
        byte_string = [i2b_vocab[i] for i in token_seq]
        return b"".join(byte_string).decode('utf-8', errors='replace')

    def find_pairs(self, token_seq: list[list[int]]) -> dict[tuple[int, int], int]:
        pairs: dict[tuple[int, int], int] = defaultdict(int)
        for seq in token_seq:
            for i in range(len(seq)-1):
                pairs[(seq[i], seq[i+1])] += 1
        return pairs

    def train(
            self, 
            token_seq: list[list[int]], 
            i2b_vocab: dict[int, bytes], 
            # b2i_vocab: dict[bytes, int], 
            total_merges: int = 50
        ) -> tuple[list[int], dict[int, bytes], dict[bytes, int], list[tuple[tuple[int, int], int]]]:

        assert isinstance(token_seq, list) and all(isinstance(x, list) for x in token_seq)

        merge_order: list[tuple[tuple[int, int], int]] = []

        for _ in range(total_merges):
            pairs: dict[tuple[int, int], int] = self.find_pairs(token_seq)
            max_pair: tuple[int, int] = max(pairs, key=lambda k: (pairs[k], k))
            v_idx: int = max(i2b_vocab) + 1
            b_string: bytes = i2b_vocab[max_pair[0]] + i2b_vocab[max_pair[1]]

            merge_order.append((max_pair, v_idx))
            
            i2b_vocab[v_idx] = b_string
            # b2i_vocab[b_string] = v_idx
            
            token_seq_copy: list[list[int]] = token_seq.copy()

            for seq in token_seq_copy:
                for i in range(len(seq)-1, 0, -1):
                    if (seq[i-1], seq[i]) == max_pair:
                        seq[i] = v_idx
                        del seq[i-1]
            token_seq = token_seq_copy
            del token_seq_copy

        # return token_seq, i2b_vocab, b2i_vocab, merge_order
        return token_seq, i2b_vocab, merge_order
        

In [13]:
pattern = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

def pre_tokenization(string: str) -> list[str]:
    return re.findall(pattern, string)

In [14]:
def mp_regex(text: list[str], num_workers: int = os.cpu_count()) -> list[str]:
    chunksize = max(1, len(text) // (num_workers * 4))
    
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        results: list[list[str]] = list(executor.map(pre_tokenization, text, chunksize=chunksize))
    results: list[str] = list(chain.from_iterable(results))

In [15]:

def main(text, i2b_vocab=i2b_vocab, total_merges=3, pattern=None):
    start = time()
    if not pattern:
        pattern = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
    # pre_tokens = re.findall(pattern, text)
    pre_tokens = mp_regex(text)
    del text
    end = time()
    print(f"{'Multiprocessing RegEx Done in':<50}: {end-start}", 
          f"Starting Tokenizer Training"
         )
    
    pre_tokens = [list(map(int, x.encode('utf-8'))) for x in pre_tokens]
    
    tokenizer = RegexTokenizer()
    pairs = tokenizer.find_pairs(pre_tokens)

    token_seq, i2b_vocab, merge_order = tokenizer.train(pre_tokens, i2b_vocab, total_merges=total_merges)
    print(f"{'Total time taken for training':<50}: {end_start}")
    return token_seq, i2b_vocab, merge_order
    

In [ ]:
example_text = """
low low low low low
lower lower widest widest widest
newest newest newest newest newest newest
"""

token_seq, i2b_vocab, merge_order = main(text, total_merges=(10000-257), pattern=pattern)

In [ ]:
decoded_seq = RegexTokenizer().nested_decode(token_seq, i2b_vocab)
print(decoded_seq == example_text)

In [ ]:
encoded_seq = RegexTokenizer().encode(example_text, merge_order)
print(encoded_seq)

In [ ]:
flat_decoded_seq = RegexTokenizer().flattened_decode(encoded_seq, i2b_vocab)
print(flat_decoded_seq)
print(flat_decoded_seq == example_text)